# Ultra-Low-Power Atrial Fibrillation Detection via Quantized TCN on Microcontrollers
**Student:** Bektur Asilov · **Project #1** (AI / IoT Sensors)

**Notebook 1 of the project: data → model → compression → export for the ESP32-S3.**

| Step | What happens |
|---|---|
| 1 | Load the **MIT-BIH Atrial Fibrillation Database** (PhysioNet): 23 ECG recordings, ~10 h each, 250 Hz, rhythm labels by cardiologists |
| 2 | **Model A — raw ECG:** 10-s ECG windows → TCN (with data augmentation) |
| 3 | **Model B — RR intervals:** a light R-peak detector (the same code runs on the microcontroller) → sequences of 32 heartbeat intervals → TCN |
| 4 | Split **by patient**: test patients are never seen in training |
| 5 | Measure **sensitivity, specificity, accuracy, F1, ROC-AUC** and compare A vs B |
| 6 | Compress the best model: float32 → **int8 TFLite** (runs on ESP32-S3) → simulated **6 / 4 / 3 / 2-bit** weights |
| 7 | Export the int8 model as a C header for the firmware |

**Research question:** can sub-byte quantization keep sensitivity above 97 % while the model fits in a microcontroller's memory?

> **Kaggle settings:** *Add Input* → dataset `marufhs/atrial-fibrillation`; *Session options* → **Internet: On** (for `pip install wfdb`).

In [ ]:
import os, glob, time, random, contextlib, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, sosfilt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

FS_ORIG = 250            # sampling rate of the database (Hz)
FS      = 125            # rate used on the microcontroller (Hz)
WIN     = 10 * FS        # model A: 10-s window = 1250 samples
N_RR    = 32             # model B: 32 heartbeat intervals per sequence
RR_STRIDE = 8            # new sequence every 8 beats
DB_DIR  = '/kaggle/working/afdb'
OUT_DIR = '/kaggle/working'
LABELS  = {'(N': 0, '(AFIB': 1}
print('TensorFlow', tf.__version__)

## 1. Data
`wfdb` is the official PhysioNet library for reading ECG records. The Kaggle dataset **"Atrial Fibrillation (MIT BIH 2000)"** is an exact copy of the PhysioNet files; if it is not attached, the records are downloaded from PhysioNet (slow).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb

attached = [d for d in {os.path.dirname(p) for p in glob.glob('/kaggle/input/**/04015.dat', recursive=True)}
            if '/old' not in d]
if attached:
    DB_DIR = attached[0]
    print('Using attached Kaggle dataset:', DB_DIR)
else:
    os.makedirs(DB_DIR, exist_ok=True)
    for rec in wfdb.get_record_list('afdb'):
        if not os.path.exists(os.path.join(DB_DIR, rec + '.dat')):
            try:
                wfdb.dl_database('afdb', DB_DIR, records=[rec], annotators=['atr'])
            except Exception as e:
                print('skip', rec, type(e).__name__)
records = sorted(r[:-4] for r in os.listdir(DB_DIR) if r.endswith('.dat')
                 and os.path.exists(os.path.join(DB_DIR, r[:-4] + '.atr')))
print(len(records), 'records:', records)

def load(rec):
    sig = np.nan_to_num(wfdb.rdrecord(os.path.join(DB_DIR, rec), channels=[0]).p_signal[:, 0])
    ann = wfdb.rdann(os.path.join(DB_DIR, rec), 'atr')
    starts = list(ann.sample); ends = starts[1:] + [len(sig)]
    segs = [(s, e, LABELS.get(a.strip().rstrip('\x00'))) for s, e, a in zip(starts, ends, ann.aux_note)]
    return sig, segs          # segs in 250-Hz samples

DATA = {rec: load(rec) for rec in records}

## 2. Patient-wise split
Each patient (record) is only in train, only in validation or only in test, so the scores show how the model works on **new people**.

In [ ]:
def label_counts(rec):
    sig, segs = DATA[rec]
    n = sum(e - s for s, e, l in segs if l == 0); a = sum(e - s for s, e, l in segs if l == 1)
    return n, a

for s in range(200):
    rng = np.random.RandomState(SEED + s)
    recs = np.array(records); rng.shuffle(recs)
    test_r, val_r, train_r = list(recs[:5]), list(recs[5:8]), list(recs[8:])
    ok = all(sum(label_counts(r)[k] for r in grp) > 250 * 600 for grp in (test_r, val_r) for k in (0, 1))
    if ok:
        break
print('train:', train_r); print('val  :', val_r); print('test :', test_r)

## 3. Model A — TCN on the raw ECG (10-s windows)
Band-pass 0.5–40 Hz, downsample to 125 Hz, 10-s windows inside one rhythm episode, z-score per window.
To fight over-fitting to the shape of a few patients' heartbeats we add **data augmentation** (random polarity, noise, baseline drift, time shift) and **dropout**.

In [ ]:
B, A = butter(2, [0.5, 40], btype='band', fs=FS_ORIG)

def ecg_windows(rec):
    sig, segs = DATA[rec]
    x = filtfilt(B, A, sig)[::2]
    X, y = [], []
    for s, e, lab in segs:
        if lab is None: continue
        s, e = s // 2, e // 2
        for w in range(s, e - WIN + 1, WIN):
            seg = x[w:w + WIN]; sd = seg.std()
            if sd < 1e-3: continue
            X.append(((seg - seg.mean()) / sd).astype(np.float32)); y.append(lab)
    return X, y

def stack(fn, recs, channel_axis=True):
    Xs, ys = [], []
    for r in recs:
        a, b = fn(r); Xs += a; ys += b
    X = np.stack(Xs)
    return (X[..., None] if channel_axis else X), np.array(ys, np.int32)

XA_tr, yA_tr = stack(ecg_windows, train_r)
XA_va, yA_va = stack(ecg_windows, val_r)
XA_te, yA_te = stack(ecg_windows, test_r)
print('A windows  train', XA_tr.shape, 'val', XA_va.shape, 'test', XA_te.shape)

In [ ]:
def augment(x, y):
    sign = tf.where(tf.random.uniform([]) < 0.5, -1.0, 1.0)
    x = x * sign
    x = x + tf.random.normal(tf.shape(x), stddev=tf.random.uniform([], 0.0, 0.3))
    t = tf.linspace(0.0, 1.0, WIN)[:, None]
    x = x + tf.random.uniform([], 0.0, 0.8) * tf.sin(2 * np.pi * tf.random.uniform([], 0.05, 0.4) * 10 * t
                                                 + tf.random.uniform([], 0, 6.28))
    x = tf.roll(x, tf.random.uniform([], 0, WIN, tf.int32), axis=0)
    return x, y

def tcn_block(x, filters, k, dilation, drop):
    sc = x
    x = layers.Conv1D(filters, k, padding='causal', dilation_rate=dilation, activation='relu')(x)
    x = layers.Dropout(drop)(x)
    x = layers.Conv1D(filters, k, padding='causal', dilation_rate=dilation)(x)
    if sc.shape[-1] != filters:
        sc = layers.Conv1D(filters, 1)(sc)
    return layers.ReLU()(layers.Add()([x, sc]))

def build_raw_tcn():
    inp = keras.Input(shape=(WIN, 1))
    x = layers.Conv1D(8, 7, strides=2, padding='same', activation='relu')(inp)
    x = layers.MaxPooling1D(2)(x)
    for d in (1, 2, 4, 8):
        x = tcn_block(x, 16, 5, d, 0.2)
    x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    return keras.Model(inp, layers.Dense(1, activation='sigmoid')(x))

def class_weights(y):
    return {0: len(y) / (2 * max(1, (y == 0).sum())), 1: len(y) / (2 * max(1, y.sum()))}

def fit(model, ds_tr, Xv, yv, cw, epochs, lr=1e-3):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='binary_crossentropy',
                  metrics=['accuracy', keras.metrics.Recall(name='sensitivity')])
    return model.fit(ds_tr, validation_data=(Xv, yv), epochs=epochs, class_weight=cw, verbose=2,
                     callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
                                keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)])

def metrics(y_true, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {'Sensitivity': tp / (tp + fn), 'Specificity': tn / (tn + fp),
            'Accuracy': (tp + tn) / len(y_true), 'F1': f1_score(y_true, pred),
            'ROC-AUC': roc_auc_score(y_true, prob)}

ds_A = (tf.data.Dataset.from_tensor_slices((XA_tr, yA_tr)).shuffle(20000, seed=SEED)
        .map(augment, num_parallel_calls=tf.data.AUTOTUNE).batch(256).prefetch(2))
model_A = build_raw_tcn()
t0 = time.time()
hist_A = fit(model_A, ds_A, XA_va, yA_va, class_weights(yA_tr), epochs=15)
pA = model_A.predict(XA_te, batch_size=512, verbose=0).ravel()
res_A = metrics(yA_te, pA)
print(f'Model A trained in {time.time() - t0:.0f} s')
print('Model A (raw ECG) on test patients:', {k: round(v, 4) for k, v in res_A.items()})

## 4. Model B — TCN on heartbeat intervals (RR)
AFib is defined by an **irregularly irregular** rhythm, so the time between heartbeats (RR interval) carries most of the information and does not depend on the shape of a particular patient's ECG.

The R-peak detector below is deliberately simple — it is **exactly the same algorithm** that runs in the ESP32-S3 firmware (causal band-pass → derivative² → 150 ms moving average → adaptive threshold with 250 ms refractory period), so the model sees the same kind of input on the chip.

Each input is a sequence of **32 RR intervals** with two channels:
1. `(RR − 0.8 s) / 0.25` — heart rate information;
2. `(RR / mean(RR) − 1) / 0.15` — irregularity, independent of heart rate.

In [ ]:
SOS = butter(2, [0.5, 40], btype='band', fs=FS_ORIG, output='sos')
MA = 19                                   # 150 ms at 125 Hz
DECAY = 0.5 ** (1 / (2.0 * FS))           # peak tracker halves in 2 s
THR, REFR, WARM = 0.35, int(0.25 * FS), 2 * FS

try:
    from numba import njit
except ImportError:
    njit = lambda f: f

@njit
def detect_r(x):
    # x: causally filtered ECG at 125 Hz -> indices of detected beats
    out = np.empty(len(x) // 20 + 10, np.int64); n = 0
    buf = np.zeros(MA); ssum = 0.0; prev = x[0]; pk = 0.0; last = -10**9; armed = True
    for i in range(len(x)):
        d = x[i] - prev; prev = x[i]
        e = d * d
        ssum += e - buf[i % MA]; buf[i % MA] = e
        env = ssum / MA
        pk = max(pk * DECAY, env)
        if i < WARM: continue
        if armed and env > THR * pk and i - last > REFR:
            out[n] = i; n += 1; last = i; armed = False
            if n >= len(out): break
        elif not armed and env < 0.5 * THR * pk:
            armed = True
    return out[:n]

def rr_features(rr):
    rr = np.clip(rr, 0.2, 2.0)
    return np.stack([(rr - 0.8) / 0.25, (rr / rr.mean() - 1) / 0.15], -1).astype(np.float32)

def rr_sequences(rec):
    sig, segs = DATA[rec]
    x = sosfilt(SOS, sig)[::2]
    beats = detect_r(x)                        # 125-Hz indices
    rr = np.diff(beats) / FS
    lab_at = np.full(len(x), -1, np.int8)
    for s, e, l in segs:
        if l is not None: lab_at[s // 2:e // 2] = l
    X, y = [], []
    for k in range(0, len(rr) - N_RR + 1, RR_STRIDE):
        a, b = beats[k], beats[k + N_RR]
        labs = lab_at[a:b]
        if labs.min() < 0 or labs.min() != labs.max(): continue   # must lie inside one N or AFIB episode
        X.append(rr_features(rr[k:k + N_RR])); y.append(int(labs[0]))
    return X, y

t0 = time.time()
XB_tr, yB_tr = stack(rr_sequences, train_r, channel_axis=False)
XB_va, yB_va = stack(rr_sequences, val_r, channel_axis=False)
XB_te, yB_te = stack(rr_sequences, test_r, channel_axis=False)
print(f'RR extraction {time.time() - t0:.0f} s | train', XB_tr.shape, 'val', XB_va.shape, 'test', XB_te.shape,
      '| AFib share test', round(yB_te.mean(), 3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for i, (lab, name) in enumerate([(0, 'Normal rhythm'), (1, 'Atrial fibrillation')]):
    seq = XB_te[np.where(yB_te == lab)[0][len(np.where(yB_te == lab)[0]) // 2]]
    ax[i].plot(seq[:, 0] * 0.25 + 0.8, 'o-'); ax[i].set_title(name); ax[i].set_xlabel('beat #')
ax[0].set_ylabel('RR interval (s)'); plt.tight_layout(); plt.show()

In [ ]:
def build_rr_tcn():
    inp = keras.Input(shape=(N_RR, 2))
    x = inp
    for d in (1, 2, 4, 8):
        x = tcn_block(x, 16, 3, d, 0.1)
    x = layers.GlobalAveragePooling1D()(x)
    return keras.Model(inp, layers.Dense(1, activation='sigmoid')(x))

ds_B = tf.data.Dataset.from_tensor_slices((XB_tr, yB_tr)).shuffle(50000, seed=SEED).batch(256).prefetch(2)
model_B = build_rr_tcn()
model_B.summary()
t0 = time.time()
hist_B = fit(model_B, ds_B, XB_va, yB_va, class_weights(yB_tr), epochs=40)
pB = model_B.predict(XB_te, batch_size=1024, verbose=0).ravel()
res_B = metrics(yB_te, pB)
print(f'Model B trained in {time.time() - t0:.0f} s')
print('Model B (RR) on test patients:', {k: round(v, 4) for k, v in res_B.items()})

## 5. Comparison on the test patients

In [ ]:
cmp_tbl = pd.DataFrame([dict(Model='A: TCN on raw ECG (10 s)', Params=model_A.count_params(), **res_A),
                        dict(Model='B: TCN on RR intervals (32 beats)', Params=model_B.count_params(), **res_B)])
display(cmp_tbl.round(4))
best_name = 'B' if res_B['F1'] >= res_A['F1'] else 'A'
print('Model chosen for the microcontroller:', best_name)
model, X_tr, X_te, y_te, p_te = ((model_B, XB_tr, XB_te, yB_te, pB) if best_name == 'B'
                                 else (model_A, XA_tr, XA_te, yA_te, pA))

## 6. Compression (quantization)
* **int8** — TensorFlow Lite full-integer model (weights and activations 8-bit); this exact file runs on the ESP32-S3 with TFLite Micro.
* **6 / 4 / 3 / 2-bit** — not supported by TFLite Micro, so they are **simulated**: every weight is rounded to 2ⁿ levels (per output channel) and the accuracy is measured.

In [ ]:
def to_tflite(m, Xrep, int8=False):
    conv = tf.lite.TFLiteConverter.from_keras_model(m)
    if int8:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        rep = Xrep[np.random.RandomState(0).choice(len(Xrep), min(1000, len(Xrep)), replace=False)]
        conv.representative_dataset = lambda: ([s[None]] for s in rep)
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return conv.convert()

def run_tflite(blob, Xd, bs=1024):
    outs = []
    for i in range(0, len(Xd), bs):
        xb = Xd[i:i + bs]
        it = tf.lite.Interpreter(model_content=blob)
        inp = it.get_input_details()[0]
        it.resize_tensor_input(inp['index'], [len(xb), *Xd.shape[1:]]); it.allocate_tensors()
        inp, out = it.get_input_details()[0], it.get_output_details()[0]
        if inp['dtype'] == np.int8:
            sc, zp = inp['quantization']; xb = np.clip(np.round(xb / sc + zp), -128, 127).astype(np.int8)
        it.set_tensor(inp['index'], xb); it.invoke()
        o = it.get_tensor(out['index']).astype(np.float32)
        if out['dtype'] == np.int8:
            sc, zp = out['quantization']; o = (o - zp) * sc
        outs.append(o.ravel())
    return np.concatenate(outs)

def fake_quantize(m, bits):
    q = keras.models.clone_model(m); q.set_weights(m.get_weights())
    levels = 2 ** (bits - 1) - 1
    for layer in q.layers:
        if isinstance(layer, (layers.Conv1D, layers.Dense)):
            w = layer.get_weights(); k = w[0]
            scale = np.max(np.abs(k), axis=tuple(range(k.ndim - 1)), keepdims=True) / levels
            scale[scale == 0] = 1
            w[0] = np.round(k / scale).clip(-levels, levels) * scale
            layer.set_weights(w)
    return q

n_params = model.count_params()
tfl_f32 = to_tflite(model, X_tr)
tfl_i8 = to_tflite(model, X_tr, int8=True)
rows = [dict(Model='Keras float32', Bits=32, **metrics(y_te, p_te), Weights_KB=n_params * 4 / 1024),
        dict(Model='TFLite int8 (runs on ESP32)', Bits=8, **metrics(y_te, run_tflite(tfl_i8, X_te)),
             Weights_KB=n_params / 1024)]
for b in (6, 4, 3, 2):
    pq = fake_quantize(model, b).predict(X_te, batch_size=1024, verbose=0).ravel()
    rows.append(dict(Model=f'Simulated {b}-bit weights', Bits=b, **metrics(y_te, pq), Weights_KB=n_params * b / 8 / 1024))
results = pd.DataFrame(rows)
results['Meets_97%_sensitivity'] = results['Sensitivity'] >= 0.97
print('Model', best_name, '| params', n_params, '| float32 TFLite', round(len(tfl_f32) / 1024, 1),
      'KB | int8 TFLite', round(len(tfl_i8) / 1024, 1), 'KB')
results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(results.Bits.astype(str), results.Sensitivity * 100, 'o-', label='Sensitivity')
ax.plot(results.Bits.astype(str), results.Specificity * 100, 's-', label='Specificity')
ax.axhline(97, ls='--', c='gray', label='97 % target')
ax.set_xlabel('bits per weight'); ax.set_ylabel('%'); ax.set_title('Accuracy vs. compression (model ' + best_name + ')')
ax.legend(); plt.show()

## 7. Export for the ESP32-S3
`afib_model_data.h` is included by the firmware. `model_config.txt` tells the firmware which model it is (input type).

In [ ]:
it = tf.lite.Interpreter(model_content=tfl_i8); it.allocate_tensors()
print('Ops needed on ESP32:', sorted({o['op_name'] for o in it._get_ops_details()} - {'DELEGATE'}))
print('Input:', it.get_input_details()[0]['shape'], it.get_input_details()[0]['quantization'])
print('Output:', it.get_output_details()[0]['quantization'])

def write_header(blob, path, kind):
    with open(path, 'w') as f:
        f.write('// AFib detector, int8 TFLite model (generated by the Kaggle notebook)\n#pragma once\n')
        f.write(f'#define AFIB_MODEL_KIND_{kind} 1\n')
        f.write('alignas(16) const unsigned char afib_model[] = {\n')
        for i in range(0, len(blob), 12):
            f.write('  ' + ', '.join(f'0x{b:02x}' for b in blob[i:i + 12]) + ',\n')
        f.write(f'}};\nconst unsigned int afib_model_len = {len(blob)};\n')

write_header(tfl_i8, f'{OUT_DIR}/afib_model_data.h', 'RR' if best_name == 'B' else 'RAW')
open(f'{OUT_DIR}/afib_int8.tflite', 'wb').write(tfl_i8)
model.save(f'{OUT_DIR}/afib_tcn.keras')
results.round(4).to_csv(f'{OUT_DIR}/quantization_results.csv', index=False)
cmp_tbl.round(4).to_csv(f'{OUT_DIR}/model_comparison.csv', index=False)
open(f'{OUT_DIR}/test_records.txt', 'w').write(','.join(test_r))
# raw-ECG model too, for the thesis comparison
write_header(to_tflite(model_A, XA_tr, int8=True), f'{OUT_DIR}/afib_raw_model_data.h', 'RAW')
print('Saved to', OUT_DIR, os.listdir(OUT_DIR))

## Conclusions
Filled in from the tables above:
1. **Model A (raw ECG TCN)** and **Model B (RR-interval TCN)** — sensitivity / specificity on unseen patients: see section 5.
2. **int8** model size and the change in sensitivity: see section 6.
3. **Sub-byte:** the lowest bit-width that still keeps sensitivity ≥ 97 %: see section 6.

**Next step:** run `afib_model_data.h` on the ESP32-S3 (firmware `AFibMonitor`) and measure inference time, RAM and energy on the chip.